### Objective

This notebook translates customer behavior and value insights into actionable segments for marketing and CRM teams. CLV modeling provides continuous value measures; segmentation groups customers into interpretable categories that support targeted communication, retention strategies, and budget allocation.

Two complementary approaches are applied:

1. **RFM-based scoring** offers transparent, interpretable categorization based on recency, frequency, and monetary value. These segments are easy to explain to non-technical stakeholders.
2. **K-Means clustering** provides a data-driven view by identifying inherent groupings across multiple behavioral dimensions. This captures nuanced patterns not apparent through rule-based segmentation alone.

By combining these methods, the analysis balances interpretability with analytical rigor, producing segments that are operationally viable and behaviorally meaningful.

### Outputs

- Customer-level segmentation table with RFM scores and cluster assignments.
- Segment-level summary with behavioral profiles and recommended actions.
- Both exported for downstream targeting, uplift simulations, and BI reporting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Color palette
COLOR_PRIMARY = "#2E4057"
COLOR_SECONDARY = "#E67E22"
COLOR_ACCENT = "#27AE60"
COLOR_GRID = "#D5D8DC"
COLOR_TEXT = "#2C3E50"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": COLOR_TEXT,
    "axes.labelcolor": COLOR_TEXT,
    "text.color": COLOR_TEXT,
    "xtick.color": COLOR_TEXT,
    "ytick.color": COLOR_TEXT,
    "grid.color": COLOR_GRID,
    "grid.alpha": 0.4,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Load datasets
customer_features_path = "/kaggle/input/datasets/customer-features/customers_features.csv"
customer_features = pd.read_csv(customer_features_path, parse_dates = ["FirstPurchase", "LastPurchase"])

probabilistic_clv12_path = "/kaggle/input/customers-clv-prob-12m/customers_clv_probabilistic_12m.csv"
probabilistic_clv12 = pd.read_csv(probabilistic_clv12_path)

ml_clv6_path = "/kaggle/input/customers-clv-ml-6m/customers_clv_ml_6m.csv"
ml_clv6 = pd.read_csv(ml_clv6_path)

# Merge CLV estimates into customer features
df = customer_features.merge(
    probabilistic_clv12[["CustomerID", "CLV_12m"]], 
    on = "CustomerID", 
    how = "left"
).merge(
    ml_clv6[["CustomerID", "CLV_6m_pred"]], 
    on = "CustomerID", 
    how = "left"
)

# Validate column names after merge
print(f"Columns in merged dataset: {df.columns.tolist()}")
print(f"Total customers: {len(df):,}")
print(f"Customers with probabilistic CLV: {df['CLV_12m'].notna().sum():,}")
print(f"Customers with ML CLV: {df['CLV_6m_pred'].notna().sum():,}")
print(f"Customers missing both CLV estimates: {df['CLV_12m'].isna().sum():,}")

Columns in merged dataset: ['CustomerID', 'Frequency', 'Monetary', 'AvgOrderValue', 'AvgItemsPerOrder', 'FirstPurchase', 'LastPurchase', 'RecencyDays', 'TenureDays', 'CustomerAgeDays', 'PurchaseRateMonthly', 'CLV_12m', 'CLV_6m_pred']
Total customers: 5,840
Customers with probabilistic CLV: 4,171
Customers with ML CLV: 4,932
Customers missing both CLV estimates: 1,669


## Data Integration and Coverage

The merged dataset contains customer features from the Customer 360 table alongside both CLV estimates. Coverage varies across value measures:

- **5,840 customers** have feature data (RFM metrics).
- **4,171 customers** have probabilistic CLV (repeat buyers only).
- **4,932 customers** have ML CLV (pre-cutoff customers only).
- **1,669 customers** lack probabilistic CLV (one-time buyers).

These coverage gaps are structural and not errors. One-time buyers cannot have their spending distribution modeled by Gamma-Gamma, and customers acquired after the ML cutoff date have no historical features. The segmentation handles these customers using RFM scores (available for all) while treating CLV as a supplementary dimension for customers where estimates exist.

## RFM Scoring for Interpretable Segments

RFM scoring translates continuous behavioral metrics into clear, interpretable segments. Customers are assigned to quintiles (1-5) on each dimension:

- **Recency (R):** Inverted so higher scores indicate more recent purchase.
- **Frequency (F):** Higher scores indicate more orders.
- **Monetary (M):** Higher scores indicate greater total spend.

Quintile-based scoring ranks customers relative to each other rather than against arbitrary thresholds. This makes the segmentation robust to scale differences and adaptable to the specific distribution of customer behavior.

Composite RFM scores (R×100 + F×10 + M) are mapped to business-aligned segment labels using a rule-based system that prioritizes actionable categories.

In [2]:
# RFM quintile scoring
# Recency is inverted so recent customers receive higher scores
df["R_score"] = pd.qcut(-df["RecencyDays"], 5, labels = [1, 2, 3, 4, 5]).astype(int)
df["F_score"] = pd.qcut(df["Frequency"].rank(method = "first"), 5, labels = [1, 2, 3, 4, 5]).astype(int)
df["M_score"] = pd.qcut(df["Monetary"].rank(method = "first"), 5, labels = [1, 2, 3, 4, 5]).astype(int)

df["RFM_Score"] = df["R_score"] * 100 + df["F_score"] * 10 + df["M_score"]

# Segment classification rules
def rfm_label(r, f, m):
    """Map RFM scores to business-aligned segment labels."""
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 4 and f >= 3:
        return "Loyal Customers"
    if r >= 4 and f <= 2:
        return "New/Promising"
    if r <= 2 and f >= 3:
        return "At Risk"
    if r <= 2 and f <= 2 and m >= 3:
        return "Need Attention"
    return "Hibernating/Low Value"

df["RFM_Segment"] = [rfm_label(r, f, m) for r, f, m in zip(df["R_score"], df["F_score"], df["M_score"])]

# Distribution
segment_counts = df["RFM_Segment"].value_counts()
print("\nRFM Segment Distribution:")
for segment, count in segment_counts.items():
    pct = count / len(df) * 100
    print(f"  {segment}: {count:,} ({pct:.1f}%)")


RFM Segment Distribution:
  Hibernating/Low Value: 2,446 (41.9%)
  Champions: 1,275 (21.8%)
  At Risk: 827 (14.2%)
  Loyal Customers: 611 (10.5%)
  New/Promising: 440 (7.5%)
  Need Attention: 241 (4.1%)


## RFM Segment Distribution: Interpretation

The RFM segmentation reveals clear stratification across the customer base:

- **Hibernating/Low Value (41.9%):** The largest segment, representing customers with limited recent engagement and low purchase frequency. This is expected in retail - a substantial portion of customers make one or few purchases and naturally lapse.
- **Champions (21.8%):** High-value, frequent, and recent customers. This segment is the strategic priority for retention and loyalty programs.
- **At Risk (14.2%):** Customers with strong historical frequency but declining recency. These are prime candidates for reactivation campaigns.
- **Loyal Customers (10.5%):** Consistent repeat buyers who may be encouraged to increase spend or frequency.
- **New/Promising (7.5%):** Recently active customers with limited purchase history. These represent future growth potential.
- **Need Attention (4.1%):** Lapsed customers with meaningful historical value. Targeted outreach may recover value.

The uneven distribution confirms that customer value and engagement are not uniformly distributed, reinforcing the need for differentiated, segment-specific strategies.

## K-Means Clustering for Behavioral Grouping

K-Means clustering identifies inherent groupings based on multiple behavioral dimensions, uncovering patterns not captured by RFM bands alone.

**Feature scaling:** Distance-based algorithms require comparable scales. Monetary value and AOV have much larger ranges than recency or purchase rate. Standardization ensures each feature contributes proportionally.

**Cluster count:** k = 4 balances analytical detail with operational utility. Four clusters distinguish broad customer archetypes without creating fragments too small to operationalize.

**Features selected:**
- RecencyDays (engagement freshness).
- Frequency (purchase count).
- Monetary (total spend).
- AvgOrderValue (spend per transaction).
- PurchaseRateMonthly (purchase intensity).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

cluster_features = df[["RecencyDays", "Frequency", "Monetary", "AvgOrderValue", "PurchaseRateMonthly"]].copy()

# Standardize features for distance-based clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_features)

# Fit K-Means with k = 4
k = 4
kmeans = KMeans(n_clusters = k, random_state = 42, n_init = "auto")
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Distribution
cluster_counts = df["Cluster"].value_counts().sort_index()
print("\nK-Means Cluster Distribution:")
for cluster, count in cluster_counts.items():
    pct = count / len(df) * 100
    print(f"  Cluster {cluster}: {count:,} ({pct:.1f}%)")


K-Means Cluster Distribution:
  Cluster 0: 3,832 (65.6%)
  Cluster 1: 1,976 (33.8%)
  Cluster 2: 12 (0.2%)
  Cluster 3: 20 (0.3%)


In [4]:
# Cluster profiling with comprehensive metrics
cluster_profile = df.groupby("Cluster").agg(
    Customers = ("CustomerID", "nunique"),
    AvgRecency = ("RecencyDays", "mean"),
    AvgFrequency = ("Frequency", "mean"),
    AvgMonetary = ("Monetary", "mean"),
    AvgAOV = ("AvgOrderValue", "mean"),
    AvgPurchaseRate = ("PurchaseRateMonthly", "mean"),
    AvgCLV12 = ("CLV_12m", "mean"),
    AvgCLV6 = ("CLV_6m_pred", "mean")
).round(2)

cluster_profile

,Customers,AvgRecency,AvgFrequency,AvgMonetary,AvgAOV,AvgPurchaseRate,AvgCLV12,AvgCLV6
Cluster,,,,,,,,
0,3832,66.79,7.63,3093.37,380.52,0.81,1821.53,1021.08
1,1976,460.87,2.21,670.97,300.51,1.07,232.46,206.87
2,12,310.83,2.92,908.02,235.95,31.08,616.16,1144.70
3,20,24.85,147.25,166137.74,1597.90,6.45,69974.25,36800.23


In [5]:
# Map clusters to business labels with profile descriptions
cluster_label_map = {
    3: "VIP/Whales",
    2: "Bulk/Business Buyers",
    0: "Core Customers",
    1: "Low-Value/Dormant"
}

df["Cluster_Label"] = df["Cluster"].map(cluster_label_map).fillna("Unclassified")

# Verify cluster labeling
print("\nCluster Label Distribution:")
print(df["Cluster_Label"].value_counts())


Cluster Label Distribution:
Cluster_Label
Core Customers          3832
Low-Value/Dormant       1976
VIP/Whales                20
Bulk/Business Buyers      12
Name: count, dtype: int64


### K-Means Cluster Interpretation

The clustering reveals four distinct behavioral archetypes with significant variation in customer count:

**Cluster 0: Core Customers (65.6%)**
- Moderate recency (67 days average), frequency (7.6 orders), and spend (£3,093).
- Average purchase rate of 0.81 orders per month.
- Balanced CLV profile (£1,822 long-term, £1,021 short-term).
- The operational backbone of the business with steady, predictable purchasing.

**Cluster 1: Low-Value/Dormant (33.8%)**
- Long recency (461 days average), minimal frequency (2.2 orders), low spend (£671).
- Purchase rate of 1.07 orders per month reflects the zero-tenure artifact for one-time buyers.
- Low CLV estimates (£232 long-term, £207 short-term).
- Natural churn and one-time purchasers; best addressed through low-cost automated reactivation.

**Cluster 2: High-Frequency Specialists (0.2%)**
- Only 12 customers with extraordinary purchase rate (31.1 orders per month).
- Moderate monetary value (£908) but very high transaction frequency.
- Strong short-term CLV (£1,145) despite weaker long-term signal (£616).
- Likely represents bulk buyers or business accounts with recurring small orders.

**Cluster 3: VIP/Whales (0.3%)**
- 20 customers with exceptional metrics across all dimensions.
- Average monetary value of £166,138 and frequency of 147 orders.
- Average long-term CLV of £69,974 and short-term CLV of £36,800.
- Recent engagement (25 days average recency).
- Disproportionate revenue contribution warranting dedicated relationship management.

The extreme imbalance reflects genuine retail dynamics: most customers follow moderate engagement patterns, while a small number show extreme behaviors with outsized strategic importance.

## Segment Summary and Recommended Actions

Each segment is summarized using key behavioral and value metrics to enable comparison at a strategic level. Long-term probabilistic CLV and short-term ML CLV provide complementary views of value, supporting nuanced prioritization.

In [6]:
# Segment-level summary with both CLV measures
segment_summary = df.groupby("RFM_Segment", as_index = False).agg(
    Customers = ("CustomerID", "nunique"),
    AvgCLV12 = ("CLV_12m", "mean"),
    AvgCLV6 = ("CLV_6m_pred", "mean"),
    AvgMonetary = ("Monetary", "mean"),
    AvgRecency = ("RecencyDays", "mean"),
    AvgFrequency = ("Frequency", "mean"),
    MedianCLV12 = ("CLV_12m", "median"),
    MedianCLV6 = ("CLV_6m_pred", "median")
).sort_values("AvgCLV12", ascending = False)

segment_summary.round(2)

,RFM_Segment,Customers,AvgCLV12,AvgCLV6,AvgMonetary,AvgRecency,AvgFrequency,MedianCLV12,MedianCLV6
1,Champions,1275,4280.42,2262.79,8953.31,19.30,17.07,2111.47,1106.37
5,New/Promising,440,938.81,324.99,542.07,27.91,1.46,526.86,188.50
3,Loyal Customers,611,883.76,369.44,992.20,24.16,3.82,573.10,289.30
2,Hibernating/Low Value,2446,879.67,351.22,1036.77,292.28,3.00,555.49,164.19
0,At Risk,827,381.38,419.86,1740.53,368.30,5.01,266.62,277.81
4,Need Attention,241,336.71,531.91,1152.20,413.68,1.59,304.52,286.83


### Segment-Level Insights: Patterns Worth Noting

The dual CLV perspective reveals several patterns that warrant explicit attention:

**Need Attention: Higher Near-Term Potential Than Expected**

The Need Attention segment shows the highest short-term CLV (£532) among all non-premium segments, exceeding even At Risk (£420) and Loyal Customers (£369). Despite long recency (414 days) and low frequency (1.6 orders), these customers are predicted to generate meaningful near-term revenue. This suggests the ML model has identified latent signals - perhaps product category preferences, or purchasing patterns - that indicate higher re-engagement likelihood than their RFM scores alone would suggest.

**Strategic implication:** These customers may deserve higher priority than their RFM classification implies. A targeted reactivation test on this segment could yield disproportionate returns relative to investment.

**At Risk: Immediate Revenue Opportunity**

The At Risk segment shows short-term CLV (£420) exceeding long-term CLV (£381). This inversion is unusual - most segments show long-term value exceeding near-term predictions. It indicates that despite lapsed engagement (368 days average recency), these customers are predicted to spend meaningfully in the next six months. Their historical frequency (5 orders) suggests established purchasing habits that may re-activate with the right stimulus.

**Strategic implication:** Time-sensitive win-back campaigns should prioritize this segment. The window for recovery may be closing as recency continues to increase.

**Hibernating/Low Value: Hidden Value in the Largest Segment**

The Hibernating segment's median CLV12 (£555) is nearly double its median CLV6 (£164). This gap suggests that while most customers in this segment are genuinely low-value, a subset retains meaningful long-term potential that the short-term model does not capture. The segment's size (2,446 customers, 41.9% of the base) masks internal heterogeneity.

**Strategic implication:** This segment could be split in future iterations. A simple frequency threshold (e.g., separating one-time buyers from customers with 2-3 orders) could reveal a sub-group worth reactivation investment.

**Cluster Size Limitations**

The Bulk/Business Buyers cluster contains only 12 customers, and VIP/Whales contains 20. While these segments are analytically distinct and economically significant, their small size limits operational targeting. Campaigns designed for these segments will reach very few customers, making it difficult to measure statistically significant uplift.

**Strategic implication:** These clusters should be treated as key accounts requiring individual relationship management rather than programmatic campaigns. For measurement purposes, they could be combined with adjacent segments when running A/B tests.

### Recommended Actions by Segment

**Champions (Top Priority)**
- Dedicated account management.
- Premium loyalty rewards and early access.
- Referral programs to leverage advocacy.
- Goal: Retain and maximize lifetime value.

**Need Attention (Elevated Priority)**
- Prioritize reactivation tests given high short-term CLV signals.
- Investigate what the ML model has identified as re-engagement potential.
- Test personalized win-back offers against this segment specifically.
- Goal: Convert latent near-term value into realized revenue.

**At Risk (Retention Priority - Time Sensitive)**
- Win-back campaigns with targeted offers.
- Satisfaction surveys to identify churn drivers.
- Personalized outreach before full lapse.
- Goal: Restore engagement within 90 days before recency extends further.

**Loyal Customers (Growth Opportunity)**
- Upsell and cross-sell campaigns.
- Tiered loyalty progression toward Champion status.
- Personalized product recommendations.
- Goal: Increase frequency and spend.

**New/Promising (Development Focus)**
- Onboarding sequences and education.
- First-repeat-purchase incentives.
- Habit-forming engagement.
- Goal: Convert to Loyal Customers.
 
**Hibernating/Low Value (Selective Approach)**
- Split segment internally: distinguish one-time buyers from 2-3 purchase customers.
- Automated email win-back for the full segment.
- Test reactivation on the 2-3 purchase sub-group.
- Accept natural attrition among one-time buyers.
- Goal: Minimal-cost reactivation with selective investment where latent value signals exist.

**Bulk/Business Buyers and VIP/Whales (Key Account Management)**
- Individual relationship management rather than programmatic campaigns.
- Dedicated account managers for top 20 VIPs.
- Custom pricing and service agreements for business buyers.
- Goal: Retain and grow revenue from outsized-value accounts.

## Export Segmentation Results

Two output levels are exported:

1. **Customer-level table:** Individual RFM scores, segments, cluster assignments, and CLV estimates for precise targeting and CRM integration.
2. **Segment-level summary:** Aggregated profiles for stakeholder communication and dashboarding.

In [7]:
# Customer-level segmentation export
customers_segments = df[[
    "CustomerID", "RFM_Segment", "RFM_Score", "Cluster", "Cluster_Label",
    "RecencyDays", "Frequency", "Monetary", "AvgOrderValue", "PurchaseRateMonthly",
    "CLV_12m", "CLV_6m_pred"
]].copy()

customers_segments_file = "customer_segments.csv"
customers_segments.to_csv(customers_segments_file, index = False)
print(f"Saved: {customers_segments_file}")
print(f"Records: {len(customers_segments):,}")

# Segment-level summary export
segment_summary_file = "segment_summary.csv"
segment_summary.to_csv(segment_summary_file, index = False)
print(f"Saved: {segment_summary_file}")
print(f"Segments: {len(segment_summary)}")

Saved: customer_segments.csv
Records: 5,840
Saved: segment_summary.csv
Segments: 6


## Segmentation Limitations and Future Refinement

This segmentation provides a solid foundation for customer targeting, but several refinements could improve operational effectiveness in future iterations (should more data be available, where required):

1. **Split Hibernating/Low Value** by purchase frequency to distinguish one-time buyers from customers with repeat purchase history but lapsed engagement.
2. **Incorporate product category data** to enable category-specific reactivation campaigns.
3. **Add engagement signals** beyond transactions - email opens, site visits, or app usage - to improve short-term CLV predictions for recently inactive customers.
4. **Re-cluster with log-transformed monetary features** to potentially reveal more balanced clusters than the current extreme separation between core and VIP segments.
5. **Validate segment stability** by re-running the segmentation on different time windows to confirm that segment boundaries are not artifacts of the specific observation period.